In [79]:
from pathlib import Path
import json, re

folder = Path("/path/3_answer_qa/3_qa_file_output_folder")
files = sorted(
    folder.glob("part_*.jsonl"),
    key=lambda f: int(re.search(r"part_(\d+)\.jsonl", f.name).group(1))
)

all_records = []
for fp in files:
    with fp.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            all_records.append(json.loads(line))


In [80]:
len(all_records),all_records[0]

(299929,
 [{'image_path': 'h_cluster:s3://mllm/internvl_data/internvl_data_image_video/langchao2other/multi-modal/Super-CLEVR/images/superCLEVR_new_004806.png',
   'qa_list': [{'question': 'Which vehicle in the image has a blue body with red wheels?\n   - A) The green pickup truck\n   - B) The yellow car\n   - C) The blue car\n   - D) The gray car',
     'answer': 'C'},
    {'question': 'How many bicycles are present in the image?\n   - A) One\n   - B) Two\n   - C) Three\n   - D) Four',
     'answer': 'B'},
    {'question': 'What type of aircraft is depicted in the image?\n   - A) Helicopter\n   - B) Jet plane\n   - C) Propeller plane\n   - D) Fighter jet',
     'answer': 'D'},
    {'question': 'Which vehicle is positioned closest to the fighter jet?\n   - A) The yellow car\n   - B) The green pickup truck\n   - C) The blue car\n   - D) The gray car',
     'answer': 'A'},
    {'question': 'What is the color of the motorcycle in the image?\n   - A) Green\n   - B) Red\n   - C) Blue\n   - 

In [81]:
final = []
v_acc = []
n_acc = []
import copy
import torch
from tqdm import tqdm

# obtain the samples that visual output is correct but nlp output is wrong
for sample in tqdm(all_records[:]):
    if not len(sample):
        continue
    qas = sample[0]['qa_list']
    visual_acc = torch.Tensor(sample[1]).view(-1 , 4).mean(dim=-1)
    nlp_acc = torch.Tensor(sample[2]).view(-1 , 4).mean(dim=-1)

    new_qas = []
    for idx in range(visual_acc.shape[0]):
        if visual_acc[idx] >=0.75 and nlp_acc[idx] <= 0.25:
            new_qas.append(qas[idx])
    if len(new_qas) < 2:
        continue

    new_sample = {
        "image_path":sample[0]['image_path'],
        "qas": new_qas,
        "ori_qas": qas
    }
    final.append(new_sample)


  0%|          | 0/299929 [00:00<?, ?it/s]

100%|██████████| 299929/299929 [00:44<00:00, 6805.90it/s]
